# blocking-eval-gpu
Generated from `code/business_entity_resolution/src/run_blocking_eval.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/blocking.py
"""Candidate generation (blocking): union of several searches, per country.

Methods (all per country; the country is a compute split, 0 true pairs cross countries):
- name       forward TF-IDF top-k on core_name
- name_city  forward TF-IDF top-k on core_name + city
- name_addr  forward TF-IDF top-k on core_name + normalised address
- reverse    for every S2/S3 record, its top-m S1 on core_name + address (all S1 of the split compete)
- rare       S2/S3 records sharing a rare core_name token (document frequency <= RARE_DF in the country)

TF-IDF: char_wb 3-4-grams, fitted per country on train + test records (no labels; a seeded sample
of up to FIT_SAMPLE texts). N-grams in more than MAX_DF of the texts are dropped: they carry almost no
weight and dominate the cost of the sparse product.

Search scope (a compute split, not a filter):
- "state": a forward query searches the pool records of its own state plus all pool records without a
  state; an S1 without a state searches the whole country. The reverse search runs within states only
  (records without a state are reached by the forward searches).
- "country": every search covers the whole country (affordable on a GPU); recovers pairs whose states
  disagree.
The rare-token index ignores states in both scopes.

Top-k search runs on the GPU when torch sees one (dense score blocks + torch.topk), otherwise on CPU with
sparse_dot_topn. check_gpu() compares the two on a slice before a run (tie-aware: scores per rank and
exact recomputation) and the faster device does the searches.

Near-identical decoys are NOT filtered here; every method keeps its full top-k.
"""
from __future__ import annotations

import time
from collections.abc import Callable

import numpy as np
import polars as pl
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

SEED = 42
FIT_SAMPLE = 3_000_000
MAX_DF = 0.02
RARE_DF = 20
NGRAMS = (3, 4)
FORWARD = ("name", "name_city", "name_addr")
REVERSE_REP = "name_addr"
NONE = "__none__"
ALL = "__all__"
SCOPES = ("state", "country")
N_THREADS = 4


def texts(df: pl.DataFrame) -> pl.DataFrame:
    """Text representations used by the searches (all from norm-v3 columns)."""
    core = pl.col("core_name")
    return df.select(
        core.alias("name"),
        pl.concat_str([core, pl.col("city").fill_null("")], separator=" ").str.strip_chars().alias("name_city"),
        pl.concat_str([core, pl.col("addr_norm").str.replace_all(",", "")], separator=" ").str.strip_chars().alias("name_addr"),
    )


def fit_vectorizer(corpus: pl.Series) -> TfidfVectorizer:
    if corpus.len() > FIT_SAMPLE:
        corpus = corpus.sample(FIT_SAMPLE, seed=SEED)
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=NGRAMS, min_df=2, max_df=MAX_DF,
                        sublinear_tf=True, dtype=np.float32)
    v.fit(corpus.to_list())
    return v


try:  # GPU path (Kaggle GPU); falls back to sparse_dot_topn on CPU
    import torch
    USE_GPU = torch.cuda.is_available()
except ImportError:
    torch, USE_GPU = None, False
GPU_BLOCK_BYTES = 2 * 1024**3  # cap for each dense block per step (scores: chunk x pool; queries: chunk x vocab)


def gpu_info() -> str:
    if not USE_GPU:
        return "no GPU (CPU sparse_dot_topn)"
    return f"torch {torch.__version__}, {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB"


def _to_torch_csr(M: sparse.csr_matrix, device: str):
    return torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int32)), torch.from_numpy(M.indices.astype(np.int32)),
                                   torch.from_numpy(M.data.astype(np.float32)), size=M.shape, device=device)


def _topn_gpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None, device: str = "cuda"):
    """Top-k of Q @ P.T per row on the GPU: P stays on the card as CSR; each query chunk is densified,
    multiplied (cuSPARSE SpMM) and torch.topk keeps the best k pool rows per query."""
    Pg = _to_torch_csr(P, device)
    chunk = int(max(1, min(4096, GPU_BLOCK_BYTES // (4 * P.shape[0]), GPU_BLOCK_BYTES // (4 * Q.shape[1]))))
    rows, cols, vals, ranks = [], [], [], []
    for s in range(0, Q.shape[0], chunk):
        Qc = _to_torch_csr(Q[s:s + chunk], device).to_dense()      # chunk x V
        scores = torch.sparse.mm(Pg, Qc.T).T.contiguous()           # chunk x pool
        v, i = torch.topk(scores, k, dim=1)                         # chunk x k, sorted descending
        v, i = v.cpu().numpy().reshape(-1), i.cpu().numpy().reshape(-1).astype(np.int64)
        n = scores.shape[0]
        r, rk = np.repeat(np.arange(s, s + n), k), np.tile(np.arange(1, k + 1), n)
        keep = v > 0  # as on CPU: only pairs sharing an n-gram
        if keep_col is not None:
            keep &= keep_col[i]
        rows.append(r[keep]); cols.append(i[keep]); vals.append(v[keep]); ranks.append(rk[keep])
        del Qc, scores
    del Pg
    return tuple(np.concatenate(x) for x in (rows, cols, vals, ranks))


def _topn_cpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None):
    R = sp_matmul_topn(Q, P.T.tocsr(), top_n=k, threshold=0.0, sort=True, n_threads=N_THREADS).tocsr()
    counts = np.diff(R.indptr)
    rows = np.repeat(np.arange(R.shape[0]), counts)
    ranks = np.arange(R.nnz) - np.repeat(R.indptr[:-1], counts) + 1
    cols = R.indices.astype(np.int64)
    keep = np.ones(R.nnz, bool) if keep_col is None else keep_col[cols]
    return rows[keep], cols[keep], R.data.astype(np.float32)[keep], ranks[keep]


def _topn(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None):
    """Top-k columns of Q @ P.T per row: (row, col, score, rank 1..k). keep_col: optional mask over P rows;
    pairs whose pool row is not kept are dropped after ranking (ranks are unchanged)."""
    if Q.shape[0] == 0 or P.shape[0] == 0:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.float32), e
    k = min(k, P.shape[0])
    return (_topn_gpu if USE_GPU else _topn_cpu)(Q, P, k, keep_col)


def check_gpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, n_q: int = 4000, n_p: int = 1_000_000) -> dict:
    """GPU vs CPU top-k on a slice, tie-aware. Many pool records share a name, so equal scores at the k-th
    place may be kept differently; instead of comparing (row, col) pairs, compare the score at every rank
    (rank_score_diff) and recompute each GPU pair's cosine exactly on CPU (pair_score_diff). Timing runs after
    a warm-up call, so CUDA start-up is not counted."""
    Q, P = Q[:n_q], P[:n_p]
    k = min(k, P.shape[0])
    _topn_gpu(Q[:64], P, k); torch.cuda.synchronize()  # warm-up (CUDA context, cuSPARSE handles)
    t0 = time.time(); g = _topn_gpu(Q, P, k); torch.cuda.synchronize(); t1 = time.time()
    c = _topn_cpu(Q, P, k); t2 = time.time()
    by_rank = lambda x: pl.DataFrame({"r": x[0], "rk": x[3], "v": x[2]})
    j = by_rank(c).join(by_rank(g), on=["r", "rk"], how="full", suffix="_g").with_columns(pl.col("v", "v_g").fill_null(0.0))
    exact = np.asarray(Q[g[0]].multiply(P[g[1]]).sum(axis=1)).ravel()
    gp, cp = set(zip(g[0].tolist(), g[1].tolist())), set(zip(c[0].tolist(), c[1].tolist()))
    pairs = Q.shape[0] * P.shape[0]
    return {"queries": Q.shape[0], "pool": P.shape[0], "k": k,
            "rank_score_diff": float((j["v"] - j["v_g"]).abs().max() or 0.0),
            "pair_score_diff": float(np.abs(exact - g[2]).max()) if len(exact) else 0.0,
            "pair_agreement": len(gp & cp) / max(len(cp), 1),
            "gpu_ns_per_pair": (t1 - t0) / pairs * 1e9, "cpu_ns_per_pair": (t2 - t1) / pairs * 1e9}


def use_gpu(flag: bool) -> None:
    """Choose the device for all later searches (GPU only if CUDA is available)."""
    global USE_GPU
    USE_GPU = bool(flag) and torch is not None and torch.cuda.is_available()


def buckets(s1: pl.DataFrame, pool: pl.DataFrame, scope: str) -> tuple[np.ndarray, np.ndarray]:
    if scope == "country":
        return np.full(s1.height, ALL, dtype=object), np.full(pool.height, ALL, dtype=object)
    return s1["state"].fill_null(NONE).to_numpy(), pool["state"].fill_null(NONE).to_numpy()


def forward(Q, q_bucket: np.ndarray, P, p_bucket: np.ndarray, k: int) -> pl.DataFrame:
    """For each query row: top-k pool rows among its state bucket + pool rows without a state."""
    out = []
    p_none = np.flatnonzero(p_bucket == NONE)
    for b in np.unique(q_bucket):
        qi = np.flatnonzero(q_bucket == b)
        pi = np.arange(P.shape[0]) if b == NONE else np.concatenate([np.flatnonzero(p_bucket == b), p_none])
        r, c, s, rk = _topn(Q[qi], P[pi], k)
        out.append(pl.DataFrame({"qi": qi[r], "pi": pi[c], "score": s, "rank": rk}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32, "rank": pl.Int64})


def reverse(S, s_bucket: np.ndarray, P, p_bucket: np.ndarray, m: int, keep_s: np.ndarray | None = None) -> pl.DataFrame:
    """For each pool row with a state: its top-m S1 rows in the same state. Returns (qi = S1 row, pi, score, rank).
    keep_s: optional mask over S1 rows; only pairs with a kept S1 are returned (all S1 still compete)."""
    out = []
    for b in np.unique(p_bucket):
        if b == NONE:
            continue
        pi = np.flatnonzero(p_bucket == b)
        si = np.flatnonzero(s_bucket == b)
        r, c, s, rk = _topn(P[pi], S[si], m, None if keep_s is None else keep_s[si])
        out.append(pl.DataFrame({"qi": si[c], "pi": pi[r], "score": s, "rank": rk}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32, "rank": pl.Int64})


def rare_tokens(q_names: pl.Series, p_names: pl.Series, df_counts: pl.DataFrame, max_df: int = RARE_DF) -> pl.DataFrame:
    """Pool rows sharing a rare core_name token with the query (token df <= max_df in the country).
    score = number of shared rare tokens; rank by score (ties by pool order)."""
    rare = df_counts.filter((pl.col("df") <= max_df) & (pl.col("t").str.len_chars() >= 3)).select("t")
    tok = lambda s, name: (pl.DataFrame({name: np.arange(s.len()), "t": s.str.split(" ")})
                           .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null()).unique()
                           .join(rare, on="t"))
    qt, pt = tok(q_names, "qi"), tok(p_names, "pi")
    hits = qt.join(pt, on="t").group_by("qi", "pi").agg(pl.len().cast(pl.Float32).alias("score"))
    return hits.with_columns(
        pl.col("score").rank("ordinal", descending=True).over("qi").cast(pl.Int64).alias("rank")
    ).select("qi", "pi", "score", "rank")


def token_df(names: pl.Series) -> pl.DataFrame:
    """Document frequency of core_name tokens (distinct per record)."""
    return (pl.DataFrame({"i": np.arange(names.len()), "t": names.str.split(" ")})
            .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null() & (pl.col("t") != "")).unique()
            .group_by("t").agg(pl.len().alias("df")))


def block_country(s1: pl.DataFrame, pool: pl.DataFrame, query_mask: np.ndarray, fit_corpus: dict[str, pl.Series],
                  df_counts: pl.DataFrame, k: int, m: int, scopes: tuple[str, ...] = ("state",),
                  skip: Callable[[str, str, float], bool] = lambda scope, method, pairs: False,
                  log: Callable[[str], None] = print, gpu_check: bool = False,
                  timing: list[dict] | None = None) -> tuple[pl.DataFrame, list[dict]]:
    """All methods for one country, for each search scope (same TF-IDF matrices). s1: ALL S1 of the split
    (reverse competition); query_mask: S1 rows to generate candidates for. skip(scope, method, pairs) may
    drop a search (e.g. over a time budget). Returns a long table (scope, qi = S1 row, pi = pool row,
    method, score, rank) and one timing row per (scope, method), appended to `timing` if given (so skip()
    can read the rates measured so far)."""
    ts, tp = texts(s1), texts(pool)
    qrows = np.flatnonzero(query_mask)
    bk = {sc: buckets(s1, pool, sc) for sc in scopes}
    parts, timing = [], ([] if timing is None else timing)
    for rep in FORWARD:
        t0 = time.time()
        v = fit_vectorizer(fit_corpus[rep])
        S, P = v.transform(ts[rep].to_list()), v.transform(tp[rep].to_list())
        t1 = time.time()
        log(f"  {rep}: fit+transform {t1 - t0:.0f}s (vocab {len(v.vocabulary_):,}, pool nnz/row {P.nnz / max(P.shape[0], 1):.1f})")
        timing.append({"scope": "all", "method": f"{rep}_fit_transform", "search_s": t1 - t0, "queries": len(qrows),
                       "pool": P.shape[0], "pairs_evaluated": None})
        if gpu_check and USE_GPU:
            chk = check_gpu(S[qrows], P, k)
            log(f"  GPU check: {chk}")
            if chk["rank_score_diff"] > 1e-4 or chk["pair_score_diff"] > 1e-4:
                raise RuntimeError(f"GPU top-k disagrees with CPU: {chk}")
            use_gpu(chk["gpu_ns_per_pair"] < chk["cpu_ns_per_pair"])  # the faster device does all searches
            log(f"  search device from now on: {'GPU' if USE_GPU else 'CPU (faster than GPU on this data)'}")
            timing.append({"scope": "all", "method": "device_check", **chk, "device": "GPU" if USE_GPU else "CPU"})
            gpu_check = False
        for sc in scopes:
            s_bucket, p_bucket = bk[sc]
            pairs = _pairs_evaluated(s_bucket[qrows], p_bucket)
            if skip(sc, rep, pairs):
                log(f"  [{sc}] {rep}: skipped ({pairs:.2e} pairs over budget)")
                timing.append({"scope": sc, "method": rep, "pairs_evaluated": pairs, "skipped": True})
            else:
                t2 = time.time()
                f = forward(S[qrows], s_bucket[qrows], P, p_bucket, k)
                f = f.with_columns(pl.Series("qi", qrows[f["qi"].to_numpy()]))  # query-local row -> S1 row
                parts.append(f.with_columns(pl.lit(sc).alias("scope"), pl.lit(rep).alias("method")))
                timing.append({"scope": sc, "method": rep, "search_s": time.time() - t2, "queries": len(qrows),
                               "pool": P.shape[0], "pairs_evaluated": pairs})
                log(f"  [{sc}] {rep}: forward {time.time() - t2:.0f}s, {pairs:.2e} pairs, {f.height:,} rows")
            if rep == REVERSE_REP:
                pairs = _pairs_evaluated(p_bucket[p_bucket != NONE], s_bucket, with_none=False)
                if skip(sc, "reverse", pairs):
                    log(f"  [{sc}] reverse: skipped ({pairs:.2e} pairs over budget)")
                    timing.append({"scope": sc, "method": "reverse", "pairs_evaluated": pairs, "skipped": True})
                    continue
                t3 = time.time()
                r = reverse(S, s_bucket, P, p_bucket, m, keep_s=query_mask)
                parts.append(r.with_columns(pl.lit(sc).alias("scope"), pl.lit("reverse").alias("method")))
                timing.append({"scope": sc, "method": "reverse", "search_s": time.time() - t3,
                               "queries": int((p_bucket != NONE).sum()), "pool": S.shape[0], "pairs_evaluated": pairs})
                log(f"  [{sc}] reverse: {time.time() - t3:.0f}s, {pairs:.2e} pairs, {r.height:,} rows")
        del S, P
    t5 = time.time()
    rr = rare_tokens(s1["core_name"][qrows], pool["core_name"], df_counts)
    rr = rr.with_columns(pl.Series("qi", qrows[rr["qi"].to_numpy()], dtype=pl.Int64), pl.lit("rare").alias("method"))
    parts += [rr.with_columns(pl.lit(sc).alias("scope")) for sc in scopes]  # scope-independent
    timing.append({"scope": "all", "method": "rare", "search_s": time.time() - t5, "queries": len(qrows),
                   "pool": pool.height, "pairs_evaluated": None})
    log(f"  rare: {time.time() - t5:.0f}s")
    cols = ["scope", "qi", "pi", "method", "score", "rank"]
    return pl.concat([p.select(cols) for p in parts]), timing


def _pairs_evaluated(q_bucket: np.ndarray, p_bucket: np.ndarray, with_none: bool = True) -> float:
    """Number of (query, pool) similarity evaluations implied by the bucket sizes (cost model)."""
    pb = pl.Series(p_bucket).value_counts()
    sizes = dict(zip(pb[pb.columns[0]].to_list(), pb["count"].to_list()))
    n_none = sizes.get(NONE, 0) if with_none else 0
    total = 0.0
    for b, n in zip(*np.unique(q_bucket, return_counts=True)):
        total += n * (len(p_bucket) if b == NONE else sizes.get(b, 0) + n_none)
    return total


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_blocking_eval.py', '--norm-dir', '/kaggle/input', '--data-dir', '/kaggle/input', '--split', '/kaggle/input', '--out-dir', '/kaggle/working', '--budget-min', '420']

In [ ]:
"""Blocking evaluation on the validation split against the FULL train S2/S3 pool of each country.

Usage: python src/run_blocking_eval.py --norm-dir artifacts/normalised --data-dir data_parquet \
           --split outputs/eda/g25_split.parquet --out-dir artifacts/blocking_eval
--norm-dir / --split may be folders to search (e.g. /kaggle/input). Outputs in --out-dir:
  val_candidates.parquet  one row per (S1, candidate): rank and score per method (null = not found)
  recall_by_method.csv, recall_union_grid.csv, recall_groups.csv, timing.csv, test_estimate.csv, blocking_report.md
Every table has a `scope` column: "state" (searches within state buckets) or "country" (whole country).
Country-scope searches whose projected time would pass --budget-min are skipped and reported.
"""
import argparse
import json
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    sys.path.insert(0, "/tmp/src")
import blocking  # noqa: E402
from blocking import FORWARD, NONE, SCOPES, _pairs_evaluated, block_country, buckets, gpu_info, texts, token_df  # noqa: E402

K_GRID = [5, 10, 20, 30, 50, 75, 100]
M_GRID = [0, 1, 2, 3, 5, 10]
METHODS = [*FORWARD, "reverse", "rare"]
COLS = ["entity_id", "country", "business_name", "core_name", "city", "addr_norm", "state"]
NON_LATIN = r"[\p{L}&&[^\p{Latin}]]"
T0 = time.time()


def log(msg: str) -> None:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    print(f"[{time.time() - T0:7.1f}s peak {peak:5.1f} GB] {msg}", flush=True)


def find(root: Path, name: str, parent: str | None = None) -> Path:
    if root.is_file():
        return root
    hits = sorted(p for p in root.rglob(name) if parent is None or p.parent.name == parent)
    if not hits:
        raise FileNotFoundError(f"{name} (parent {parent}) not found under {root}")
    return hits[0]


def md(df: pl.DataFrame, max_rows: int = 60) -> str:
    df = df.head(max_rows)
    fmt = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(fmt(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def test_pairs(te: dict, country: str, scopes: list[str]) -> list[dict]:
    """Similarity evaluations a test run would need for this country, per scope (same cost model as timing)."""
    t1 = te[1].filter(pl.col("country") == country)
    tp = pl.concat([te[2].filter(pl.col("country") == country), te[3].filter(pl.col("country") == country)])
    out = []
    for sc in scopes:
        tsb, tpb = buckets(t1, tp, sc)
        out.append({"scope": sc, "country": country, "test_s1": t1.height, "test_pool": tp.height,
                    "forward_pairs": _pairs_evaluated(tsb, tpb),
                    "reverse_pairs": _pairs_evaluated(tpb[tpb != NONE], tsb, with_none=False)})
    return out


rates: dict[str, float] = {}  # seconds per similarity evaluation, "forward" / "reverse", measured so far


def measured_rates(timing: list[dict]) -> dict[str, float]:
    out = {}
    for kind in ("forward", "reverse"):
        rows = [t for t in timing if t.get("pairs_evaluated") and not t.get("skipped")
                and (t["method"] == "reverse") == (kind == "reverse")]
        if rows:
            out[kind] = sum(t["search_s"] for t in rows) / sum(t["pairs_evaluated"] for t in rows)
    return out


def over_budget(tim: list[dict]):
    """skip() for block_country: skip a search if its projected time (pairs x measured rate) would pass the
    budget, keeping 20% of the budget for evaluation. State-scope searches are never skipped."""
    def skip(scope: str, method: str, pairs: float) -> bool:
        if scope == "state":
            return False
        r = {**rates, **measured_rates(tim)}.get("reverse" if method == "reverse" else "forward")
        return r is not None and (time.time() - T0) + pairs * r > 0.8 * BUDGET_S
    return skip


BUDGET_S = 600 * 60.0


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--norm-dir", default="artifacts/normalised")
    ap.add_argument("--data-dir", default="data_parquet")
    ap.add_argument("--split", default="outputs/eda/g25_split.parquet")
    ap.add_argument("--out-dir", default="artifacts/blocking_eval")
    ap.add_argument("--k-max", type=int, default=max(K_GRID))
    ap.add_argument("--m-max", type=int, default=max(M_GRID))
    ap.add_argument("--scopes", nargs="+", default=list(SCOPES), choices=SCOPES)
    ap.add_argument("--budget-min", type=float, default=600, help="wall-clock budget for the searches (minutes)")
    a = ap.parse_args()
    global BUDGET_S
    BUDGET_S = a.budget_min * 60
    log(f"search device: {gpu_info()}; scopes {a.scopes}; budget {a.budget_min:.0f} min")
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    norm = lambda sp, k, cols=COLS: pl.read_parquet(find(Path(a.norm_dir), f"{sp}_source{k}.parquet", "normalised"), columns=cols)
    split = pl.read_parquet(find(Path(a.split), "g25_split.parquet"))
    val_ids = split.filter(pl.col("role") == "val")["source1_entity_id"]
    gt = pl.read_parquet(find(Path(a.data_dir), "train_ground_truth.parquet"))
    truth = (
        gt.filter(pl.col("source1_entity_id").is_in(val_ids.implode()))
        .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
        .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != ""))
        .with_columns(pl.col("cand").str.slice(0, 2).alias("msrc"))
    )
    tr = {k: norm("train", k) for k in (1, 2, 3)}
    te = {k: norm("test", k, ["country", "core_name", "city", "addr_norm", "state"]) for k in (1, 2, 3)}
    log(f"loaded; {val_ids.len():,} validation S1, {truth.height:,} validation true pairs")

    cands, timing, test_est, attrs = [], [], [], []
    for country in tr[1]["country"].unique().sort().to_list():
        log(f"country {country}")
        s1 = tr[1].filter(pl.col("country") == country)
        pool = pl.concat([tr[2].filter(pl.col("country") == country), tr[3].filter(pl.col("country") == country)])
        test_all = pl.concat([t.filter(pl.col("country") == country) for t in te.values()], how="diagonal_relaxed")
        train_all = pl.concat([s1, pool])
        corpus = {rep: pl.concat([texts(train_all)[rep], texts(test_all)[rep]]) for rep in FORWARD}
        df_counts = token_df(pl.concat([train_all["core_name"], test_all["core_name"]]))
        qmask = s1["entity_id"].is_in(val_ids.implode()).to_numpy()
        tim: list[dict] = []
        long, _ = block_country(s1, pool, qmask, corpus, df_counts, a.k_max, a.m_max, scopes=tuple(a.scopes),
                                skip=over_budget(tim), log=log, gpu_check=not timing, timing=tim)
        cands.append(long.with_columns(pl.Series("s1", s1["entity_id"].to_numpy()[long["qi"].to_numpy()]),
                                       pl.Series("cand", pool["entity_id"].to_numpy()[long["pi"].to_numpy()])).drop("qi", "pi"))
        timing += [{"country": country, **t} for t in tim]
        rates.update(measured_rates(timing))
        # test-time cost from the test bucket sizes of this country (same cost model as timing)
        test_est += test_pairs(te, country, a.scopes)
        # attributes of validation S1 for group recall
        cnt = pl.concat([train_all["core_name"], test_all["core_name"]]).alias("core_name").value_counts(name="n_core")
        attrs.append(s1.filter(pl.Series(qmask)).select(pl.col("entity_id").alias("s1"), "country", "core_name")
                     .join(cnt, on="core_name", how="left"))
        del s1, pool, test_all, train_all, corpus
        log(f"country {country} done")
    # France etc. (test-only countries) still need a test-time estimate
    for country in sorted(set(te[1]["country"].unique().to_list()) - set(tr[1]["country"].unique().to_list())):
        test_est += test_pairs(te, country, a.scopes)

    long = pl.concat(cands)
    del cands
    wide = long.pivot(on="method", index=["scope", "s1", "cand"], values=["rank", "score"], aggregate_function="min")
    del long
    for mth in METHODS:
        for c in ("rank", "score"):
            if f"{c}_{mth}" not in wide.columns:
                wide = wide.with_columns(pl.lit(None, dtype=pl.Int64 if c == "rank" else pl.Float32).alias(f"{c}_{mth}"))
    wide = wide.join(truth.select("s1", "cand", pl.lit(True).alias("is_match")), on=["s1", "cand"], how="left").with_columns(
        pl.col("is_match").fill_null(False))
    wide.write_parquet(out / "val_candidates.parquet")
    log(f"candidates: {wide.height:,} (scope, S1, cand) rows at k={a.k_max}, m={a.m_max}")

    val_s1 = pl.concat(attrs)
    n_s1 = val_s1.group_by("country").len("n_s1")
    t = truth.join(val_s1.select("s1", "country"), on="s1")
    tim = pl.DataFrame(timing, infer_schema_length=None)
    tim.write_csv(out / "timing.csv")
    skipped = {(r["scope"], r["method"]) for r in tim.filter(pl.col("skipped").fill_null(False)).iter_rows(named=True)} \
        if "skipped" in tim.columns else set()

    def union(k: int, m: int, rare: bool = True) -> pl.Expr:
        e = pl.any_horizontal([pl.col(f"rank_{f}") <= k for f in FORWARD])
        if m:
            e = e | (pl.col("rank_reverse") <= m)
        if rare:
            e = e | pl.col("rank_rare").is_not_null()
        return e.fill_null(False)

    # attributes of validation true pairs for group recall
    matches = pl.concat([tr[2].select("entity_id", "core_name", "business_name", "state"),
                         tr[3].select("entity_id", "core_name", "business_name", "state")]).rename(
        {"entity_id": "cand", "core_name": "m_core", "business_name": "m_name", "state": "m_state"})
    s1_state = tr[1].select(pl.col("entity_id").alias("s1"), pl.col("state").alias("s1_state"))
    g = t.join(val_s1.select("s1", "core_name", "n_core"), on="s1").join(matches, on="cand", how="left").join(s1_state, on="s1")
    g = g.with_columns(pl.Series("tsr", process.cpdist(g["core_name"].to_list(), g["m_core"].fill_null("").to_list(),
                                                       scorer=fuzz.token_set_ratio, workers=-1)))
    groups = {
        "all": pl.lit(True),
        "common name (core_name 5+ in country)": pl.col("n_core") >= 5,
        "low name similarity (tsr < 70)": pl.col("tsr") < 70,
        "Indian-script match record": pl.col("m_name").str.contains(NON_LATIN),
        "state differs (both known)": pl.col("m_state").is_not_null() & (pl.col("m_state") != pl.col("s1_state")),
        "match has no state": pl.col("m_state").is_null(),
    }

    by_method_all, grid_all, grp_all, recs, report = [], [], [], {}, []
    for sc in a.scopes:
        w = wide.filter(pl.col("scope") == sc)

        def evaluate(sel: pl.Expr, label: dict) -> list[dict]:
            found = w.filter(sel).select("s1", "cand")
            hit = t.join(found, on=["s1", "cand"], how="semi").group_by("country").len("hit")
            size = found.join(val_s1.select("s1", "country"), on="s1").group_by("country").len("cands")
            r = (t.group_by("country").len("true").join(hit, on="country", how="left").join(size, on="country", how="left")
                 .join(n_s1, on="country").fill_null(0))
            tot = r.select(pl.lit("ALL").alias("country"), *[pl.col(c).sum() for c in ("true", "hit", "cands", "n_s1")])
            r = pl.concat([r.select(tot.columns), tot])
            return [{"scope": sc, **label, "country": x["country"], "recall": x["hit"] / max(x["true"], 1),
                     "cands_per_s1": x["cands"] / max(x["n_s1"], 1)} for x in r.iter_rows(named=True)]

        rows = []
        for mth in METHODS:
            ks = M_GRID[1:] if mth == "reverse" else ([10**9] if mth == "rare" else K_GRID)
            for k in ks:
                rows += evaluate(pl.col(f"rank_{mth}") <= k, {"method": mth, "k": k if mth != "rare" else None})
        by_method = pl.DataFrame(rows)
        rows = []
        for k in K_GRID:
            for m in M_GRID:
                rows += evaluate(union(k, m), {"k": k, "m": m})
        grid = pl.DataFrame(rows)

        # recommendation: cheapest (k, m) with overall union recall >= 97% (else the best recall)
        overall = grid.filter(pl.col("country") == "ALL").sort("cands_per_s1")
        ok = overall.filter(pl.col("recall") >= 0.97)
        rec = (ok if ok.height else overall.sort("recall", descending=True)).row(0, named=True)
        k_rec, m_rec = rec["k"], rec["m"]
        recs[sc] = {"k": k_rec, "m": m_rec, "recall": rec["recall"], "cands_per_s1": rec["cands_per_s1"]}

        configs = {f"recommended k={k_rec}, m={m_rec}": union(k_rec, m_rec), f"max k={a.k_max}, m={a.m_max}": union(a.k_max, a.m_max),
                   **{f"{mth} only (k={a.k_max if mth != 'reverse' else a.m_max})": pl.col(f"rank_{mth}").is_not_null() for mth in METHODS}}
        rows = []
        for cname, sel in configs.items():
            found = w.filter(sel.fill_null(False)).select("s1", "cand", pl.lit(True).alias("found"))
            gg = g.join(found, on=["s1", "cand"], how="left").with_columns(pl.col("found").fill_null(False))
            for gname, cond in groups.items():
                for country, sub in [("ALL", gg.filter(cond))] + [(c, d) for (c,), d in gg.filter(cond).group_by("country")]:
                    rows.append({"scope": sc, "config": cname, "group": gname, "country": country, "true_pairs": sub.height,
                                 "recall": sub["found"].mean() if sub.height else None})
        grp = pl.DataFrame(rows).sort("config", "group", "country")
        by_method_all.append(by_method); grid_all.append(grid); grp_all.append(grp)
        sk = sorted(m for s_, m in skipped if s_ == sc)
        report += [
            f"# Scope: {sc}\n",
            f"Recommended: **k = {k_rec}, m = {m_rec}**, union recall {rec['recall']:.4f}, {rec['cands_per_s1']:.1f} candidates per S1."
            + (f" Skipped over budget (not in the union): {', '.join(sk)}." if sk else "") + "\n",
            "## Union recall grid (ALL countries)\n", md(overall.sort("k", "m").drop("country", "scope"), 60) + "\n",
            "## Union recall per country at the recommended config\n",
            md(grid.filter((pl.col("k") == k_rec) & (pl.col("m") == m_rec)).drop("scope")) + "\n",
            "## Recall per method (ALL countries)\n", md(by_method.filter(pl.col("country") == "ALL").drop("country", "scope"), 60) + "\n",
            "## Recall by group (ALL countries)\n", md(grp.filter(pl.col("country") == "ALL").drop("country", "scope"), 80) + "\n",
        ]
        log(f"[{sc}] recommended k={k_rec}, m={m_rec}: recall {rec['recall']:.4f}, {rec['cands_per_s1']:.1f} cands/S1")
    pl.concat(by_method_all).write_csv(out / "recall_by_method.csv")
    pl.concat(grid_all).write_csv(out / "recall_union_grid.csv")
    pl.concat(grp_all).write_csv(out / "recall_groups.csv")

    # test-time estimate: seconds per similarity evaluation measured here x test pair counts
    rate = measured_rates(timing)
    fit = tim.filter(pl.col("method").str.ends_with("_fit_transform"))
    fit_rate = fit["search_s"].sum() / max(fit["pool"].sum() + tr[1].height, 1)  # seconds per record per representation
    est = pl.DataFrame(test_est).with_columns(
        (pl.col("forward_pairs") * rate.get("forward", 0.0) * len(FORWARD) / 60).alias("forward_min"),
        (pl.col("reverse_pairs") * rate.get("reverse", 0.0) / 60).alias("reverse_min"),
        ((pl.col("test_s1") + pl.col("test_pool")) * fit_rate / 60).alias("fit_transform_min"),
    ).with_columns(pl.sum_horizontal("forward_min", "reverse_min", "fit_transform_min").alias("total_min"))
    est.write_csv(out / "test_estimate.csv")
    est_tot = est.group_by("scope").agg(pl.col("^.*_min$").sum()).sort("scope")

    report += [
        "# Timing on validation (seconds)\n", md(tim, 80) + "\n",
        f"Measured rate: forward {rate.get('forward', 0) * 1e9:.2f} ns, reverse {rate.get('reverse', 0) * 1e9:.2f} ns per "
        f"similarity evaluation; searches ran on {'GPU' if blocking.USE_GPU else 'CPU'} ({gpu_info()}).\n",
        "# Test-time estimate (minutes)\n", md(est) + "\n", md(est_tot) + "\n",
        f"Peak memory of this run (host): {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.1f} GB.\n",
    ]
    (out / "blocking_report.md").write_text("\n".join(report))
    (out / "config.json").write_text(json.dumps({"recommended": recs, "k_grid": K_GRID, "m_grid": M_GRID,
                                                 "scopes": a.scopes, "device": gpu_info()}, indent=1))
    log(f"done: {recs}")


if __name__ == "__main__":
    main()
